# False Positive Benchmark Analysis

Local VS Code notebook for the real-only deepfake false-positive benchmark.

This notebook reads the centralized benchmark CSV and optional threshold sweep summary CSVs from local paths only. It does not run model inference and does not edit the main benchmark CSV.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(r"/Users/subrat/Desktop/Deepfake")
OUTPUT_BASE_DIR = PROJECT_ROOT / "output"
MODEL_FILE_PREFIX = "buildborderless__CommunityForensics-DeepfakeDet-ViT"


def latest_file(pattern: str, base_dir: Path = OUTPUT_BASE_DIR) -> Path:
    matches = sorted(base_dir.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if not matches:
        raise FileNotFoundError(f"No files matched {pattern} under {base_dir}")
    return matches[0]


MAIN_CSV_PATH = latest_file("false_positive_complete_benchmark_*.csv")
RUN_KEY = MAIN_CSV_PATH.stem.replace("false_positive_complete_benchmark_", "")
THRESHOLD_SWEEP_DIR = OUTPUT_BASE_DIR / f"threshold_sweep_summaries_{RUN_KEY}"
OUTPUT_DIR = OUTPUT_BASE_DIR / "false_positive_benchmark_analysis" / RUN_KEY

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEFAULT_THRESHOLD = 0.50
THRESHOLDS = [round(x / 100, 2) for x in range(10, 100, 5)]

print(f"MAIN_CSV_PATH: {MAIN_CSV_PATH}")
print(f"THRESHOLD_SWEEP_DIR: {THRESHOLD_SWEEP_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")


## Setup

This cell installs only the plotting and data packages needed for local analysis if they are missing. It does not install or load the deepfake model.

In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "plotly": "plotly",
    "kaleido": "kaleido",
}

missing = [package for import_name, package in REQUIRED_PACKAGES.items() if importlib.util.find_spec(import_name) is None]
if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required analysis packages are already installed.")

## Load Benchmark Data

This loads the centralized real-only benchmark CSV and prepares analysis columns from saved probability scores. Real-only labels mean `FP` is a real image predicted as fake, and `TN` is a real image predicted as real.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 180

if not MAIN_CSV_PATH.exists():
    raise FileNotFoundError(f"Main CSV not found: {MAIN_CSV_PATH}")

df = pd.read_csv(MAIN_CSV_PATH, low_memory=False)

if "source_subgroup" not in df.columns:
    df["source_subgroup"] = ""
df["raw_source"] = df.get("source", "").fillna("").astype(str)
df["source_subgroup"] = df["source_subgroup"].fillna("").astype(str)
source_subgroup_first = df["source_subgroup"].str.replace("\\", "/", regex=False).str.split("/").str[0]
source_wrapper_mask = df["raw_source"].eq("data") & source_subgroup_first.ne("")
df["source"] = df["raw_source"]
df.loc[source_wrapper_mask, "source"] = source_subgroup_first[source_wrapper_mask]


for col in ["test_type", "source", "resolution_bucket", "target_dimension", "resize_mode", "stress_type", "stress_level", "error"]:
    if col not in df.columns:
        df[col] = ""
    df[col] = df[col].fillna("").astype(str)

numeric_cols = [
    "normal_same_resolution_fake_probability",
    "normal_original_fake_probability",
    "stress_fake_probability",
    "score_delta_vs_clean",
    "score_delta_vs_original",
    "variant_width",
    "variant_height",
    "variant_megapixels",
    "original_megapixels",
    "brightness_mean",
    "contrast_std",
    "blur_score",
    "sharpness_score",
    "file_size_mb",
    "inference_ms",
]
for col in numeric_cols:
    if col not in df.columns:
        df[col] = np.nan
    df[col] = pd.to_numeric(df[col], errors="coerce")

valid = df[df["error"].eq("")].copy()
normal = valid[(valid["test_type"] == "normal") & valid["normal_same_resolution_fake_probability"].notna()].copy()
stress = valid[(valid["test_type"] == "stress") & valid["stress_fake_probability"].notna()].copy()

normal["score"] = normal["normal_same_resolution_fake_probability"]
stress["score"] = stress["stress_fake_probability"]
scored = pd.concat([normal.assign(scope="normal"), stress.assign(scope="stress")], ignore_index=True, sort=False)
scored["prediction_at_default"] = np.where(scored["score"] >= DEFAULT_THRESHOLD, "FP", "TN")

print(f"Loaded rows: {len(df):,}")
print(f"Completed normal score rows: {len(normal):,}")
print(f"Completed stress score rows: {len(stress):,}")
print(f"Default threshold: {DEFAULT_THRESHOLD}")

## Threshold Sweep Summaries

If summary CSVs already exist, this notebook loads them. If not, it computes threshold summaries from saved probability scores only. It does not run model inference and does not add rows to the centralized CSV.

In [ ]:
SWEEP_FILES = {
    "overall": THRESHOLD_SWEEP_DIR / "threshold_sweep_overall.csv",
    "by_source": THRESHOLD_SWEEP_DIR / "threshold_sweep_by_source.csv",
    "by_resolution": THRESHOLD_SWEEP_DIR / "threshold_sweep_by_resolution.csv",
    "by_source_resolution": THRESHOLD_SWEEP_DIR / "threshold_sweep_by_source_resolution.csv",
    "by_stress_type_level": THRESHOLD_SWEEP_DIR / "threshold_sweep_by_stress_type_level.csv",
}

SUMMARY_COLUMNS = [
    "threshold", "test_scope", "group_name", "group_value", "source", "resolution_bucket", "target_dimension", "resize_mode", "stress_type", "stress_level",
    "count", "FP", "TN", "FPR", "TNR", "score_mean", "score_median", "score_p95", "score_min", "score_max",
    "TN_to_TN", "TN_to_FP", "FP_to_TN", "FP_to_FP", "TN_to_TN_rate", "TN_to_FP_rate", "FP_to_TN_rate", "FP_to_FP_rate",
    "delta_vs_clean_mean", "delta_vs_clean_median", "delta_vs_clean_p95", "delta_vs_clean_min", "delta_vs_clean_max",
]

def _single_value(frame, col):
    vals = [str(v) for v in frame[col].dropna().unique().tolist() if str(v) != ""] if col in frame.columns else []
    return vals[0] if len(vals) == 1 else ""

def _metric_row(frame, threshold, scope, group_name, group_value):
    scores = pd.to_numeric(frame.get("score", pd.Series(dtype=float)), errors="coerce").dropna()
    count = int(len(scores))
    fp = int((scores >= threshold).sum()) if count else 0
    tn = count - fp
    row = {
        "threshold": threshold, "test_scope": scope, "group_name": group_name, "group_value": group_value,
        "source": _single_value(frame, "source"), "resolution_bucket": _single_value(frame, "resolution_bucket"),
        "target_dimension": _single_value(frame, "target_dimension"), "resize_mode": _single_value(frame, "resize_mode"),
        "stress_type": _single_value(frame, "stress_type"), "stress_level": _single_value(frame, "stress_level"),
        "count": count, "FP": fp, "TN": tn, "FPR": fp / count if count else np.nan, "TNR": tn / count if count else np.nan,
        "score_mean": scores.mean() if count else np.nan, "score_median": scores.median() if count else np.nan,
        "score_p95": scores.quantile(0.95) if count else np.nan, "score_min": scores.min() if count else np.nan, "score_max": scores.max() if count else np.nan,
        "TN_to_TN": np.nan, "TN_to_FP": np.nan, "FP_to_TN": np.nan, "FP_to_FP": np.nan,
        "TN_to_TN_rate": np.nan, "TN_to_FP_rate": np.nan, "FP_to_TN_rate": np.nan, "FP_to_FP_rate": np.nan,
        "delta_vs_clean_mean": np.nan, "delta_vs_clean_median": np.nan, "delta_vs_clean_p95": np.nan,
        "delta_vs_clean_min": np.nan, "delta_vs_clean_max": np.nan,
    }
    if scope == "stress" and count:
        pairs = pd.DataFrame({
            "clean": pd.to_numeric(frame["normal_same_resolution_fake_probability"], errors="coerce"),
            "stress": pd.to_numeric(frame["stress_fake_probability"], errors="coerce"),
        }).dropna()
        n = int(len(pairs))
        if n:
            clean_fp = pairs["clean"] >= threshold
            stress_fp = pairs["stress"] >= threshold
            transitions = {
                "TN_to_TN": int((~clean_fp & ~stress_fp).sum()),
                "TN_to_FP": int((~clean_fp & stress_fp).sum()),
                "FP_to_TN": int((clean_fp & ~stress_fp).sum()),
                "FP_to_FP": int((clean_fp & stress_fp).sum()),
            }
            for key, value in transitions.items():
                row[key] = value
                row[f"{key}_rate"] = value / n
        deltas = pd.to_numeric(frame["score_delta_vs_clean"], errors="coerce").dropna()
        if not deltas.empty:
            row["delta_vs_clean_mean"] = deltas.mean()
            row["delta_vs_clean_median"] = deltas.median()
            row["delta_vs_clean_p95"] = deltas.quantile(0.95)
            row["delta_vs_clean_min"] = deltas.min()
            row["delta_vs_clean_max"] = deltas.max()
    return row

def _build_summary(frame, scope, group_name, group_cols):
    rows = []
    for threshold in THRESHOLDS:
        if frame.empty:
            continue
        if not group_cols:
            rows.append(_metric_row(frame, threshold, scope, group_name, "all"))
        else:
            for keys, group in frame.groupby(group_cols, dropna=False):
                if not isinstance(keys, tuple):
                    keys = (keys,)
                row = _metric_row(group, threshold, scope, group_name, "|".join(str(k) for k in keys))
                for col, value in zip(group_cols, keys):
                    if col in row:
                        row[col] = value
                rows.append(row)
    return pd.DataFrame(rows, columns=SUMMARY_COLUMNS)

def compute_sweep_summaries():
    THRESHOLD_SWEEP_DIR.mkdir(parents=True, exist_ok=True)
    all_scored = pd.concat([normal.assign(score=normal["score"], scope="normal"), stress.assign(score=stress["score"], scope="stress")], ignore_index=True, sort=False)
    summaries = {
        "overall": pd.concat([_build_summary(normal, "normal", "overall", []), _build_summary(stress, "stress", "overall", [])], ignore_index=True),
        "by_source": pd.concat([_build_summary(normal, "normal", "source", ["source"]), _build_summary(stress, "stress", "source", ["source"])], ignore_index=True),
        "by_resolution": pd.concat([_build_summary(normal, "normal", "resolution", ["resolution_bucket"]), _build_summary(stress, "stress", "resolution", ["resolution_bucket"])], ignore_index=True),
        "by_source_resolution": pd.concat([_build_summary(normal, "normal", "source_resolution", ["source", "resolution_bucket"]), _build_summary(stress, "stress", "source_resolution", ["source", "resolution_bucket"])], ignore_index=True),
        "by_stress_type_level": _build_summary(stress, "stress", "stress_type_level", ["stress_type", "stress_level"]),
    }
    for key, frame in summaries.items():
        frame.to_csv(SWEEP_FILES[key], index=False)
    return summaries

if all(path.exists() for path in SWEEP_FILES.values()):
    sweep = {key: pd.read_csv(path) for key, path in SWEEP_FILES.items()}
    print("Loaded existing threshold sweep summaries.")
else:
    sweep = compute_sweep_summaries()
    print("Computed threshold sweep summaries from saved probability scores.")

for key, frame in sweep.items():
    print(f"{key}: {frame.shape}")

## Chart Helpers

These helper functions keep the chart sections compact. Every chart is shown inline and saved as a PNG inside `OUTPUT_DIR`.

In [ ]:
def save_current_fig(name):
    path = OUTPUT_DIR / f"{name}.png"
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.show()
    print(f"Saved: {path}")


def no_data_chart(name, message):
    plt.figure(figsize=(9, 3.5))
    plt.text(0.5, 0.5, message, ha="center", va="center", fontsize=12)
    plt.axis("off")
    save_current_fig(name)


def threshold_label(threshold=DEFAULT_THRESHOLD):
    return f"threshold >= {threshold:.2f}"

## Chart 1: 01 Score Distribution Overall

**What it shows:** Shows the distribution of fake probabilities for completed benchmark rows.

**Why it is useful:** This is useful for seeing whether real images cluster near zero or spread toward the false-positive threshold.

**Insight to look for:** Look for heavy right tails or multiple peaks.

**How to interpret results:** Higher fake probability or FPR means more real images are being misclassified as fake. Lower FPR and higher TNR are safer for this real-only benchmark.

In [ ]:
if scored.empty:
    no_data_chart("01_score_distribution_overall", "No completed score rows available.")
else:
    plt.figure(figsize=(10, 5))
    sns.histplot(data=scored, x="score", hue="scope", bins=60, kde=True, stat="density", common_norm=False)
    plt.axvline(DEFAULT_THRESHOLD, color="red", linestyle="--", label=threshold_label())
    plt.title("Fake Probability Distribution")
    plt.xlabel("Fake probability")
    plt.ylabel("Density")
    plt.legend()
    save_current_fig("01_score_distribution_overall")

## Chart 2: 02 Score Distribution By Source

**What it shows:** Shows fake probability by source category. This helps identify which real-image sources are more fragile. Look for sources with higher medians, wider boxes, or many high outliers. A source with many points above threshold has higher false-positive risk.

**Why it is useful:** This is useful for real-only false-positive analysis.

**Insight to look for:** Look for sources with higher medians, wider boxes, or many high outliers.

**How to interpret results:** Higher fake probability or FPR means more real images are being misclassified as fake. Lower FPR and higher TNR are safer for this real-only benchmark.

In [ ]:
if scored.empty or scored["source"].nunique() == 0:
    no_data_chart("02_score_by_source_box", "No scored source data available.")
else:
    plt.figure(figsize=(11, 5))
    sns.boxplot(data=scored, x="source", y="score", hue="scope")
    plt.axhline(DEFAULT_THRESHOLD, color="red", linestyle="--")
    plt.title("Fake Probability by Source")
    plt.xlabel("Source")
    plt.ylabel("Fake probability")
    plt.xticks(rotation=30, ha="right")
    save_current_fig("02_score_by_source_box")

## Chart 3: 03 Overall FPR/TNR Sweep

**What it shows:** Shows how overall false-positive rate and true-negative rate change as the threshold moves from 0.10 to 0.95.

**Why it is useful:** This is useful for choosing a safer threshold for real-only deployment.

**Insight to look for:** Look for the point where FPR becomes acceptably low while TNR remains high. Higher thresholds usually reduce false positives.

**How to interpret results:** Higher fake probability or FPR means more real images are being misclassified as fake. Lower FPR and higher TNR are safer for this real-only benchmark.

In [ ]:
overall = sweep.get("overall", pd.DataFrame()).copy()
if overall.empty:
    no_data_chart("03_overall_fpr_tnr_sweep", "No overall threshold sweep data available.")
else:
    plt.figure(figsize=(10, 5))
    for scope, part in overall.groupby("test_scope"):
        plt.plot(part["threshold"], part["FPR"], marker="o", label=f"FPR {scope}")
        plt.plot(part["threshold"], part["TNR"], marker="s", linestyle="--", label=f"TNR {scope}")
    plt.title("Overall Real-Only Threshold Sweep")
    plt.xlabel("Threshold")
    plt.ylabel("Rate")
    plt.ylim(-0.02, 1.02)
    plt.legend()
    save_current_fig("03_overall_fpr_tnr_sweep")

## Chart 4: 04 FPR Sweep By Source

**What it shows:** Shows false-positive rate by source across thresholds.

**Why it is useful:** This is useful for spotting categories that need a higher threshold than the overall average.

**Insight to look for:** Look for lines that remain high even at strict thresholds.

**How to interpret results:** Higher fake probability or FPR means more real images are being misclassified as fake. Lower FPR and higher TNR are safer for this real-only benchmark.

In [ ]:
by_source = sweep.get("by_source", pd.DataFrame()).copy()
if by_source.empty:
    no_data_chart("04_fpr_sweep_by_source", "No source threshold sweep data available.")
else:
    plt.figure(figsize=(11, 5))
    sns.lineplot(data=by_source, x="threshold", y="FPR", hue="source", style="test_scope", marker="o")
    plt.title("FPR Sweep by Source")
    plt.xlabel("Threshold")
    plt.ylabel("False positive rate")
    plt.ylim(-0.02, 1.02)
    save_current_fig("04_fpr_sweep_by_source")

## Chart 5: 05 FPR Sweep By Resolution

**What it shows:** Shows false-positive rate by resolution bucket across thresholds. This helps test whether smaller or larger images are more likely to be mislabeled fake. Look for resolution buckets that decay slowly as threshold increases. Those buckets are more sensitive to threshold choice.

**Why it is useful:** This is useful for real-only false-positive analysis.

**Insight to look for:** Look for resolution buckets that decay slowly as threshold increases.

**How to interpret results:** Higher fake probability or FPR means more real images are being misclassified as fake. Lower FPR and higher TNR are safer for this real-only benchmark.

In [ ]:
by_res = sweep.get("by_resolution", pd.DataFrame()).copy()
if by_res.empty:
    no_data_chart("05_fpr_sweep_by_resolution", "No resolution threshold sweep data available.")
else:
    plt.figure(figsize=(11, 5))
    sns.lineplot(data=by_res, x="threshold", y="FPR", hue="resolution_bucket", style="test_scope", marker="o")
    plt.title("FPR Sweep by Resolution Bucket")
    plt.xlabel("Threshold")
    plt.ylabel("False positive rate")
    plt.ylim(-0.02, 1.02)
    save_current_fig("05_fpr_sweep_by_resolution")

## Chart 6: 06 Source-Resolution FPR Heatmap

**What it shows:** Shows false-positive rate at the default threshold for each source and resolution bucket.

**Why it is useful:** This is useful for finding combined failure pockets that are hidden in overall averages.

**Insight to look for:** Look for warm cells.

**How to interpret results:** Higher fake probability or FPR means more real images are being misclassified as fake. Lower FPR and higher TNR are safer for this real-only benchmark.

In [ ]:
by_sr = sweep.get("by_source_resolution", pd.DataFrame()).copy()
heat = by_sr[np.isclose(by_sr.get("threshold", np.nan), DEFAULT_THRESHOLD)] if not by_sr.empty else pd.DataFrame()
if heat.empty:
    no_data_chart("06_source_resolution_fpr_heatmap", "No source-resolution sweep data at default threshold.")
else:
    pivot = heat.pivot_table(index="source", columns="resolution_bucket", values="FPR", aggfunc="mean")
    plt.figure(figsize=(10, 5))
    sns.heatmap(pivot, annot=True, fmt=".3f", cmap="rocket_r", vmin=0, vmax=max(0.01, np.nanmax(pivot.values)))
    plt.title(f"FPR by Source and Resolution at Threshold {DEFAULT_THRESHOLD:.2f}")
    plt.xlabel("Resolution bucket")
    plt.ylabel("Source")
    save_current_fig("06_source_resolution_fpr_heatmap")

## Chart 7: 07 Score By Resolution Bucket

**What it shows:** Shows score distributions by resolution bucket. This helps compare whether image scale is associated with higher fake probability. Look for buckets with higher medians or many high outliers. Those buckets may need more review or a stricter threshold.

**Why it is useful:** This is useful for real-only false-positive analysis.

**Insight to look for:** Look for buckets with higher medians or many high outliers.

**How to interpret results:** Higher fake probability or FPR means more real images are being misclassified as fake. Lower FPR and higher TNR are safer for this real-only benchmark.

In [ ]:
if scored.empty:
    no_data_chart("07_score_by_resolution", "No scored resolution data available.")
else:
    plt.figure(figsize=(11, 5))
    sns.boxplot(data=scored, x="resolution_bucket", y="score", hue="scope")
    plt.axhline(DEFAULT_THRESHOLD, color="red", linestyle="--")
    plt.title("Fake Probability by Resolution Bucket")
    plt.xlabel("Resolution bucket")
    plt.ylabel("Fake probability")
    plt.xticks(rotation=25, ha="right")
    save_current_fig("07_score_by_resolution")

## Chart 8: 08 Score Vs Megapixels

**What it shows:** Shows fake probability against tested variant megapixels.

**Why it is useful:** This is useful for checking whether model confidence shifts with image size.

**Insight to look for:** Look for upward trends or clusters at certain sizes.

**How to interpret results:** Higher fake probability or FPR means more real images are being misclassified as fake. Lower FPR and higher TNR are safer for this real-only benchmark.

In [ ]:
plot_df = scored.dropna(subset=["variant_megapixels", "score"])
if plot_df.empty:
    no_data_chart("08_score_vs_megapixels", "No megapixel and score data available.")
else:
    plt.figure(figsize=(10, 5))
    sns.scatterplot(data=plot_df, x="variant_megapixels", y="score", hue="source", style="scope", alpha=0.7)
    plt.axhline(DEFAULT_THRESHOLD, color="red", linestyle="--")
    plt.title("Fake Probability vs Variant Megapixels")
    plt.xlabel("Variant megapixels")
    plt.ylabel("Fake probability")
    save_current_fig("08_score_vs_megapixels")

## Chart 9: 09 Score Vs Brightness

**What it shows:** Shows fake probability against average brightness. This helps detect whether very dark or very bright real images are more fragile. Look for high-score clusters at brightness extremes. If present, brightness normalization or threshold stratification may be worth testing.

**Why it is useful:** This is useful for real-only false-positive analysis.

**Insight to look for:** Look for high-score clusters at brightness extremes.

**How to interpret results:** Higher fake probability or FPR means more real images are being misclassified as fake. Lower FPR and higher TNR are safer for this real-only benchmark.

In [ ]:
plot_df = scored.dropna(subset=["brightness_mean", "score"])
if plot_df.empty:
    no_data_chart("09_score_vs_brightness", "No brightness and score data available.")
else:
    plt.figure(figsize=(10, 5))
    sns.scatterplot(data=plot_df, x="brightness_mean", y="score", hue="source", style="scope", alpha=0.7)
    plt.axhline(DEFAULT_THRESHOLD, color="red", linestyle="--")
    plt.title("Fake Probability vs Brightness")
    plt.xlabel("Brightness mean")
    plt.ylabel("Fake probability")
    save_current_fig("09_score_vs_brightness")

## Chart 10: 10 Score Vs Contrast

**What it shows:** Shows fake probability against contrast standard deviation.

**Why it is useful:** This is useful for seeing whether flat or high-contrast images trigger false positives.

**Insight to look for:** Look for high-score points at the low or high contrast ends.

**How to interpret results:** Higher fake probability or FPR means more real images are being misclassified as fake. Lower FPR and higher TNR are safer for this real-only benchmark.

In [ ]:
plot_df = scored.dropna(subset=["contrast_std", "score"])
if plot_df.empty:
    no_data_chart("10_score_vs_contrast", "No contrast and score data available.")
else:
    plt.figure(figsize=(10, 5))
    sns.scatterplot(data=plot_df, x="contrast_std", y="score", hue="source", style="scope", alpha=0.7)
    plt.axhline(DEFAULT_THRESHOLD, color="red", linestyle="--")
    plt.title("Fake Probability vs Contrast")
    plt.xlabel("Contrast std")
    plt.ylabel("Fake probability")
    save_current_fig("10_score_vs_contrast")

## Chart 11: 11 Score Vs Blur

**What it shows:** Shows fake probability against blur/sharpness score. This helps determine whether blurry or overly sharp real images are more often flagged. Look for high fake probabilities at low blur scores or extreme sharpness scores. That indicates sensitivity to focus or enhancement artifacts.

**Why it is useful:** This is useful for real-only false-positive analysis.

**Insight to look for:** Look for high fake probabilities at low blur scores or extreme sharpness scores. That indicates sensitivity to focus or enhancement artifacts.

**How to interpret results:** Higher fake probability or FPR means more real images are being misclassified as fake. Lower FPR and higher TNR are safer for this real-only benchmark.

In [ ]:
plot_df = scored.dropna(subset=["blur_score", "score"])
if plot_df.empty:
    no_data_chart("11_score_vs_blur", "No blur and score data available.")
else:
    plt.figure(figsize=(10, 5))
    sns.scatterplot(data=plot_df, x="blur_score", y="score", hue="source", style="scope", alpha=0.7)
    plt.axhline(DEFAULT_THRESHOLD, color="red", linestyle="--")
    plt.xscale("symlog")
    plt.title("Fake Probability vs Blur Score")
    plt.xlabel("Blur score, symlog scale")
    plt.ylabel("Fake probability")
    save_current_fig("11_score_vs_blur")

## Chart 12: 12 Score Vs File Size

**What it shows:** Shows fake probability against file size in MB when available.

**Why it is useful:** This is useful for detecting whether compression or unusually small/large files correlate with false positives.

**Insight to look for:** Look for high-score clusters at tiny file sizes or very large files. That may point to encoding or compression sensitivity.

**How to interpret results:** Higher fake probability or FPR means more real images are being misclassified as fake. Lower FPR and higher TNR are safer for this real-only benchmark.

In [ ]:
plot_df = scored.dropna(subset=["file_size_mb", "score"])
if plot_df.empty:
    no_data_chart("12_score_vs_file_size", "No file size and score data available.")
else:
    plt.figure(figsize=(10, 5))
    sns.scatterplot(data=plot_df, x="file_size_mb", y="score", hue="source", style="scope", alpha=0.7)
    plt.axhline(DEFAULT_THRESHOLD, color="red", linestyle="--")
    plt.title("Fake Probability vs File Size")
    plt.xlabel("File size MB")
    plt.ylabel("Fake probability")
    save_current_fig("12_score_vs_file_size")

## Chart 13: 13 False Positive Counts By Source

**What it shows:** Shows the number of real images predicted fake at the default threshold.

**Why it is useful:** This is useful for prioritizing manual review.

**Insight to look for:** Look for sources with the largest FP bars. These are the categories contributing most to current false-positive load.

**How to interpret results:** Higher fake probability or FPR means more real images are being misclassified as fake. Lower FPR and higher TNR are safer for this real-only benchmark.

In [ ]:
if scored.empty:
    no_data_chart("13_fp_counts_by_source", "No scored rows available for FP count.")
else:
    tmp = scored.copy()
    tmp["result_at_default"] = np.where(tmp["score"] >= DEFAULT_THRESHOLD, "FP", "TN")
    counts = tmp[tmp["result_at_default"] == "FP"].groupby(["scope", "source"]).size().reset_index(name="FP")
    if counts.empty:
        no_data_chart("13_fp_counts_by_source", f"No false positives at threshold {DEFAULT_THRESHOLD:.2f}.")
    else:
        plt.figure(figsize=(10, 5))
        sns.barplot(data=counts, x="source", y="FP", hue="scope")
        plt.title(f"False Positive Counts by Source at Threshold {DEFAULT_THRESHOLD:.2f}")
        plt.xlabel("Source")
        plt.ylabel("FP count")
        plt.xticks(rotation=30, ha="right")
        save_current_fig("13_fp_counts_by_source")

## Chart 14: 14 Stress FPR By Type And Level

**What it shows:** Shows stress false-positive rate by stress type and level at the default threshold.

**Why it is useful:** This is useful for identifying transformations that make clean real images fragile.

**Insight to look for:** Look for stress levels with high FPR. These are the controlled conditions most likely to trigger false positives.

**How to interpret results:** Higher fake probability or FPR means more real images are being misclassified as fake. Lower FPR and higher TNR are safer for this real-only benchmark.

In [ ]:
stress_tl = sweep.get("by_stress_type_level", pd.DataFrame()).copy()
stress_default = stress_tl[np.isclose(stress_tl.get("threshold", np.nan), DEFAULT_THRESHOLD)] if not stress_tl.empty else pd.DataFrame()
if stress_default.empty:
    no_data_chart("14_stress_fpr_by_type_level", "No completed stress score rows available.")
else:
    stress_default["type_level"] = stress_default["stress_type"].astype(str) + " / " + stress_default["stress_level"].astype(str)
    plt.figure(figsize=(12, 5))
    sns.barplot(data=stress_default, x="type_level", y="FPR")
    plt.title(f"Stress FPR by Type and Level at Threshold {DEFAULT_THRESHOLD:.2f}")
    plt.xlabel("Stress type / level")
    plt.ylabel("False positive rate")
    plt.xticks(rotation=35, ha="right")
    save_current_fig("14_stress_fpr_by_type_level")

## Chart 15: 15 Stress Transition Counts

**What it shows:** Shows how clean baseline decisions change after stress at the default threshold.

**Why it is useful:** This is useful for separating stable real predictions from stress-induced false positives. Look especially for `TN_to_FP`, which means a real image was safe clean but became fake after one stress factor. High `TN_to_FP` counts indicate fragile transformations.

**Insight to look for:** Look for groups with unusually high false-positive behavior.

**How to interpret results:** Higher fake probability or FPR means more real images are being misclassified as fake. Lower FPR and higher TNR are safer for this real-only benchmark.

In [ ]:
if stress.empty:
    no_data_chart("15_stress_transition_counts", "No completed stress rows available for transitions.")
else:
    tmp = stress.dropna(subset=["normal_same_resolution_fake_probability", "stress_fake_probability"]).copy()
    if tmp.empty:
        no_data_chart("15_stress_transition_counts", "Stress rows are missing clean baseline scores for transitions.")
    else:
        clean_fp = tmp["normal_same_resolution_fake_probability"] >= DEFAULT_THRESHOLD
        stress_fp = tmp["stress_fake_probability"] >= DEFAULT_THRESHOLD
        tmp["transition_at_default"] = np.select(
            [~clean_fp & ~stress_fp, ~clean_fp & stress_fp, clean_fp & ~stress_fp, clean_fp & stress_fp],
            ["TN_to_TN", "TN_to_FP", "FP_to_TN", "FP_to_FP"],
            default="unknown",
        )
        counts = tmp.groupby(["stress_type", "stress_level", "transition_at_default"]).size().reset_index(name="count")
        counts["type_level"] = counts["stress_type"].astype(str) + " / " + counts["stress_level"].astype(str)
        pivot = counts.pivot_table(index="type_level", columns="transition_at_default", values="count", fill_value=0)
        pivot[[c for c in ["TN_to_TN", "TN_to_FP", "FP_to_TN", "FP_to_FP"] if c in pivot.columns]].plot(kind="bar", stacked=True, figsize=(12, 5))
        plt.title(f"Stress Prediction Transitions at Threshold {DEFAULT_THRESHOLD:.2f}")
        plt.xlabel("Stress type / level")
        plt.ylabel("Count")
        plt.xticks(rotation=35, ha="right")
        save_current_fig("15_stress_transition_counts")

## Chart 16: 16 Delta Vs Clean By Stress Type

**What it shows:** Shows how much stress changes the fake probability relative to the matched clean baseline.

**Why it is useful:** This is useful for finding transformations that push scores upward even if they do not cross the threshold.

**Insight to look for:** Look for positive medians or high positive tails. Positive delta means stress increased fake probability compared with clean same-resolution input.

**How to interpret results:** Higher fake probability or FPR means more real images are being misclassified as fake. Lower FPR and higher TNR are safer for this real-only benchmark.

In [ ]:
plot_df = stress.dropna(subset=["score_delta_vs_clean"]).copy()
if plot_df.empty:
    no_data_chart("16_delta_vs_clean_by_stress_type", "No stress delta_vs_clean data available.")
else:
    plot_df["type_level"] = plot_df["stress_type"].astype(str) + " / " + plot_df["stress_level"].astype(str)
    plt.figure(figsize=(12, 5))
    sns.boxplot(data=plot_df, x="type_level", y="score_delta_vs_clean")
    plt.axhline(0, color="black", linestyle="--")
    plt.title("Stress Score Delta vs Clean Baseline")
    plt.xlabel("Stress type / level")
    plt.ylabel("Stress fake probability - clean fake probability")
    plt.xticks(rotation=35, ha="right")
    save_current_fig("16_delta_vs_clean_by_stress_type")

## Key Internal Insights

Run all chart cells first, then run this cell. It summarizes fragile sources, resolutions, stress types, and safe thresholds using only real-only metrics from saved scores and summaries.

In [ ]:
def top_rows(frame, sort_col="FPR", n=5):
    if frame.empty or sort_col not in frame.columns:
        return pd.DataFrame()
    return frame.sort_values([sort_col, "FP", "count"], ascending=[False, False, False]).head(n)

print("Key Internal Insights")
print("=" * 80)

source_default = sweep.get("by_source", pd.DataFrame())
source_default = source_default[np.isclose(source_default.get("threshold", np.nan), DEFAULT_THRESHOLD)] if not source_default.empty else pd.DataFrame()
print("\nFragile sources at default threshold:")
display(top_rows(source_default, "FPR", 10))

res_default = sweep.get("by_resolution", pd.DataFrame())
res_default = res_default[np.isclose(res_default.get("threshold", np.nan), DEFAULT_THRESHOLD)] if not res_default.empty else pd.DataFrame()
print("\nFragile resolution buckets at default threshold:")
display(top_rows(res_default, "FPR", 10))

stress_default = sweep.get("by_stress_type_level", pd.DataFrame())
stress_default = stress_default[np.isclose(stress_default.get("threshold", np.nan), DEFAULT_THRESHOLD)] if not stress_default.empty else pd.DataFrame()
print("\nFragile stress types and levels at default threshold:")
display(top_rows(stress_default, "FPR", 10))

safe_thresholds = []
overall = sweep.get("overall", pd.DataFrame()).copy()
if not overall.empty:
    for scope, part in overall.groupby("test_scope"):
        candidates = part[(part["FPR"] <= 0.01) & (part["count"] > 0)].sort_values("threshold")
        if not candidates.empty:
            safe_thresholds.append({"scope": scope, "first_threshold_with_FPR_at_or_below_1pct": candidates.iloc[0]["threshold"], "FPR": candidates.iloc[0]["FPR"], "TNR": candidates.iloc[0]["TNR"]})
        else:
            safe_thresholds.append({"scope": scope, "first_threshold_with_FPR_at_or_below_1pct": np.nan, "FPR": np.nan, "TNR": np.nan})

print("\nSafe threshold candidates, using FPR <= 1% rule:")
display(pd.DataFrame(safe_thresholds))

if stress.empty:
    print("\nStress insight note: No completed stress probability scores are present in the loaded CSV, so stress fragility and transition insights cannot be measured yet from this file.")
else:
    print("\nStress insight note: Check TN_to_FP and positive delta_vs_clean groups first; those are clean real images that become fragile after one controlled stress factor.")